In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Imports
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import json

from data.dataset import TwoTowerDataset

In [3]:
# Load processed train data and metadata
train_df = pd.read_csv("../data/processed/train.csv")

with open("../data/processed/metadata.json") as f:
    metadata = json.load(f)

num_movies = metadata["num_movies"]

print("train_df shape:", train_df.shape)
print("num_movies (from metadata):", num_movies)
train_df.head()

train_df shape: (569243, 4)
num_movies (from metadata): 3533


,user_id,movie_id,rating,timestamp
0,0,23,4,978300019
1,0,16,5,978300055
2,0,20,4,978300055
3,0,29,5,978300055
4,0,28,5,978300172


In [4]:
# Confirm dtypes survived the CSV round-trip as expected
train_df.dtypes

user_id      int64
movie_id     int64
rating       int64
timestamp    int64
dtype: object

In [5]:
# Construct the dataset
dataset = TwoTowerDataset(train_df, num_movies=num_movies)
print("Dataset constructed. len(dataset) =", len(dataset))

Dataset constructed. len(dataset) = 569243


In [6]:
# Test 1: __len__ matches number of positive interactions in train
assert len(dataset) == len(train_df)
print("PASS: len(dataset) == len(train_df) ->", len(dataset))

PASS: len(dataset) == len(train_df) -> 569243


In [7]:
# Test 2: __getitem__ returns plain python ints, not numpy/torch types
u, p, n = dataset[0]
print("Sample triplet:", (u, p, n))
print("Types:", type(u), type(p), type(n))

assert isinstance(u, int) and isinstance(p, int) and isinstance(n, int)
print("PASS: all returned values are plain python int")

Sample triplet: (0, 23, 1793)
Types: <class 'int'> <class 'int'> <class 'int'>
PASS: all returned values are plain python int


In [8]:
# Test 3: the positive in the triplet matches the actual row in train_df
idx = 0
u, p, n = dataset[idx]

expected_u = int(train_df.iloc[idx]["user_id"])
expected_p = int(train_df.iloc[idx]["movie_id"])

assert u == expected_u and p == expected_p
print(f"PASS: triplet positive matches train_df row {idx}")

PASS: triplet positive matches train_df row 0


In [9]:
# Test 4 (critical): no sampled negative should ever land in the user's
# train positive set. This is the entire reason _sample_negative exists.
#
# We don't just check this once -- we sweep every index, multiple times
# each, since negatives are re-sampled fresh on every __getitem__ call.

n_repeats = 3
violations = []

for trial in range(n_repeats):
    for idx in range(len(dataset)):
        u, p, n = dataset[idx]
        if n in dataset.user_positives[u]:
            violations.append((trial, idx, u, p, n))

print(f"Checked {n_repeats * len(dataset)} total draws")
print(f"Violations found: {len(violations)}")

assert len(violations) == 0, f"Found {len(violations)} negative/positive collisions!"
print("PASS: zero negatives ever collided with a user's positive set")

Checked 1707729 total draws
Violations found: 0
PASS: zero negatives ever collided with a user's positive set


In [ ]:
# Test 5: within a single triplet, negative should never equal the positive
mismatches = 0
for idx in range(len(dataset)):
    u, p, n = dataset[idx]
    if p == n:
        mismatches += 1

assert mismatches == 0
print("PASS: positive and negative are never identical within the same triplet")

In [ ]:
# Test 6: negatives are freshly sampled, not cached/static, across repeated
# calls to the same index. Confirms the rejection sampling actually runs
# each call rather than e.g. being memoized accidentally.
idx = 0
negs_seen = set()
for _ in range(50):
    _, _, n = dataset[idx]
    negs_seen.add(n)

print(f"Distinct negatives seen across 50 calls at idx={idx}: {len(negs_seen)}")
assert len(negs_seen) > 1
print("PASS: negative sampling is fresh per call")

In [ ]:
# Test 7: sampled negatives always fall within the valid movie_id range
out_of_range = 0
for idx in range(0, len(dataset), 50):  # stride for speed, full sweep not needed here
    _, _, n = dataset[idx]
    if not (0 <= n < num_movies):
        out_of_range += 1

assert out_of_range == 0
print("PASS: all sampled negatives are within [0, num_movies)")

In [ ]:
# Summary
print("All dataset.py sanity checks passed.")
print(f"  - {len(dataset)} positive interactions")
print(f"  - {len(dataset.user_positives)} users with positive sets built")
print(f"  - num_movies (sampling range): {num_movies}")